In [1]:
!wget --no-check-certificate -O /content/val2017.zip \
https://images.cocodataset.org/zips/val2017.zip

--2026-09-04 05:13:56--  https://images.cocodataset.org/zips/val2017.zip
Resolving images.cocodataset.org (images.cocodataset.org)... 16.182.66.241, 16.182.71.97, 16.182.107.89, ...
Connecting to images.cocodataset.org (images.cocodataset.org)|16.182.66.241|:443... connected.
	requested host name ‘images.cocodataset.org’.
HTTP request sent, awaiting response... 200 OK
Length: 815585330 (778M) [application/zip]
Saving to: ‘/content/val2017.zip’

/content/val2017.zi 100%[===================>] 777.80M  53.0MB/s    in 15s     

2026-09-04 05:14:12 (51.8 MB/s) - ‘/content/val2017.zip’ saved [815585330/815585330]



In [2]:
!unzip -q /content/val2017.zip -d /content/

In [3]:
!wget --no-check-certificate -O /content/annotations_trainval2017.zip \
https://images.cocodataset.org/annotations/annotations_trainval2017.zip

--2026-09-04 05:14:21--  https://images.cocodataset.org/annotations/annotations_trainval2017.zip
Resolving images.cocodataset.org (images.cocodataset.org)... 16.15.246.3, 16.182.97.193, 16.15.253.23, ...
Connecting to images.cocodataset.org (images.cocodataset.org)|16.15.246.3|:443... connected.
	requested host name ‘images.cocodataset.org’.
HTTP request sent, awaiting response... 200 OK
Length: 252907541 (241M) [application/zip]
Saving to: ‘/content/annotations_trainval2017.zip’

/content/annotation 100%[===================>] 241.19M  42.3MB/s    in 6.1s    

2026-09-04 05:14:27 (39.6 MB/s) - ‘/content/annotations_trainval2017.zip’ saved [252907541/252907541]



In [4]:
!unzip -q /content/annotations_trainval2017.zip -d /content/

In [5]:
import os

annotation_file = "/content/annotations/instances_val2017.json"

print("Annotations exist:", os.path.exists(annotation_file))

if os.path.exists(annotation_file):
    print(
        "Annotation size:",
        round(os.path.getsize(annotation_file) / (1024**2), 2),
        "MB"
    )

Annotations exist: True
Annotation size: 19.06 MB


In [6]:
!pip install -q ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 31.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.2/66.2 kB 881.9 kB/s eta 0:00:00


In [7]:
!nvidia-smi

Fri Sep  4 05:14:52 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [8]:
from ultralytics import YOLO, RTDETR

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.


In [9]:
model = YOLO("yolo11n.pt")

print("YOLO11n loaded!")

transformer_model = RTDETR("rtdetr-l.pt")

print("RT-DETR-L loaded!")

YOLO11n loaded!
RT-DETR-L loaded!


In [10]:
import json

ann_file = "/content/annotations/instances_val2017.json"

with open(ann_file, "r") as f:
    coco = json.load(f)

print("Images:", len(coco["images"]))
print("Annotations:", len(coco["annotations"]))
print("Categories:", len(coco["categories"]))

Images: 5000
Annotations: 36781
Categories: 80


In [11]:
import json
import os
from collections import defaultdict

# Your 15 classes
JETAUTO_CLASSES = [
    "person",
    "bicycle",
    "car",
    "motorcycle",
    "bus",
    "truck",
    "traffic light",
    "fire hydrant",
    "stop sign",
    "bench",
    "chair",
    "couch",
    "potted plant",
    "clock",
    "tv"
]

# Paths
COCO_JSON = "/content/annotations/instances_val2017.json"
COCO_IMAGES = "/content/val2017"

OUT_ROOT = "/content/jetauto15"
OUT_IMAGES = os.path.join(OUT_ROOT, "images", "val")
OUT_LABELS = os.path.join(OUT_ROOT, "labels", "val")

os.makedirs(OUT_IMAGES, exist_ok=True)
os.makedirs(OUT_LABELS, exist_ok=True)

# Load COCO annotations
with open(COCO_JSON, "r") as f:
    coco = json.load(f)

# Map COCO category names → COCO IDs
coco_name_to_id = {
    cat["name"]: cat["id"]
    for cat in coco["categories"]
}

# Get COCO IDs for our classes
selected_coco_ids = {
    coco_name_to_id[name]: idx
    for idx, name in enumerate(JETAUTO_CLASSES)
}

print("Selected classes:")
for name, coco_id in coco_name_to_id.items():
    if name in JETAUTO_CLASSES:
        print(f"{selected_coco_ids[coco_id]:2d} -> {name} (COCO ID {coco_id})")

Selected classes:
 0 -> person (COCO ID 1)
 1 -> bicycle (COCO ID 2)
 2 -> car (COCO ID 3)
 3 -> motorcycle (COCO ID 4)
 4 -> bus (COCO ID 6)
 5 -> truck (COCO ID 8)
 6 -> traffic light (COCO ID 10)
 7 -> fire hydrant (COCO ID 11)
 8 -> stop sign (COCO ID 13)
 9 -> bench (COCO ID 15)
10 -> chair (COCO ID 62)
11 -> couch (COCO ID 63)
12 -> potted plant (COCO ID 64)
14 -> tv (COCO ID 72)
13 -> clock (COCO ID 85)


In [12]:
from PIL import Image
import shutil

# Image ID → image information
images = {
    img["id"]: img
    for img in coco["images"]
}

# Image ID → annotations
image_annotations = defaultdict(list)

for ann in coco["annotations"]:
    if ann["category_id"] in selected_coco_ids:
        image_annotations[ann["image_id"]].append(ann)

used_images = 0
total_objects = 0

for image_id, anns in image_annotations.items():

    img_info = images[image_id]

    src_image = os.path.join(
        COCO_IMAGES,
        img_info["file_name"]
    )

    dst_image = os.path.join(
        OUT_IMAGES,
        img_info["file_name"]
    )

    # Copy image
    if not os.path.exists(dst_image):
        shutil.copy2(src_image, dst_image)

    # Create YOLO label
    label_file = os.path.join(
        OUT_LABELS,
        os.path.splitext(img_info["file_name"])[0] + ".txt"
    )

    with open(label_file, "w") as f:

        width = img_info["width"]
        height = img_info["height"]

        for ann in anns:

            x, y, w, h = ann["bbox"]

            # Convert COCO → YOLO format
            x_center = (x + w / 2) / width
            y_center = (y + h / 2) / height

            w_norm = w / width
            h_norm = h / height

            class_id = selected_coco_ids[ann["category_id"]]

            f.write(
                f"{class_id} "
                f"{x_center:.6f} "
                f"{y_center:.6f} "
                f"{w_norm:.6f} "
                f"{h_norm:.6f}\n"
            )

            total_objects += 1

    used_images += 1

print("Done!")
print("Images containing selected classes:", used_images)
print("Objects:", total_objects)

Done!
Images containing selected classes: 3527
Objects: 18499


In [13]:
print("Images:",
      len(os.listdir("/content/jetauto15/images/val")))

print("Labels:",
      len(os.listdir("/content/jetauto15/labels/val")))

Images: 3527
Labels: 3527


In [14]:
print("\nExample label:")

files = os.listdir("/content/jetauto15/labels/val")

if files:
    with open(
        "/content/jetauto15/labels/val/" + files[0],
        "r"
    ) as f:
        print(f.read())


Example label:
2 0.759523 0.272104 0.480953 0.229167
3 0.460953 0.375281 0.197188 0.426979
3 0.471812 0.619385 0.943625 0.756771
0 0.740687 0.534427 0.299656 0.817146
0 0.634547 0.355052 0.176969 0.584271



In [15]:
import yaml

data_config = {
    "path": "/content/jetauto15",
    "train": "images/val",
    "val": "images/val",
    "names": {
        0: "person",
        1: "bicycle",
        2: "car",
        3: "motorcycle",
        4: "bus",
        5: "truck",
        6: "traffic light",
        7: "fire hydrant",
        8: "stop sign",
        9: "bench",
        10: "chair",
        11: "couch",
        12: "potted plant",
        13: "clock",
        14: "tv"
    }
}

yaml_path = "/content/jetauto15.yaml"

with open(yaml_path, "w") as f:
    yaml.dump(data_config, f, sort_keys=False)

print(open(yaml_path).read())

path: /content/jetauto15
train: images/val
val: images/val
names:
  0: person
  1: bicycle
  2: car
  3: motorcycle
  4: bus
  5: truck
  6: traffic light
  7: fire hydrant
  8: stop sign
  9: bench
  10: chair
  11: couch
  12: potted plant
  13: clock
  14: tv



In [16]:
import os
import shutil

# Remove the duplicated JetAuto training images/labels
train_images = "/content/jetauto15/images/train"
train_labels = "/content/jetauto15/labels/train"

if os.path.exists(train_images):
    shutil.rmtree(train_images)
    print("Deleted duplicated training images")

if os.path.exists(train_labels):
    shutil.rmtree(train_labels)
    print("Deleted duplicated training labels")

# Remove the huge COCO training ZIP if it exists.
# We already have the extracted train2017 images.
train_zip = "/content/datasets/coco/images/train2017.zip"

if os.path.exists(train_zip):
    os.remove(train_zip)
    print("Deleted train2017.zip")

print("Cleanup complete.")

Cleanup complete.


In [17]:
!df -h /content

Filesystem      Size  Used Avail Use% Mounted on
overlay         113G   51G   63G  45% /


In [18]:
import os

for f in [
    "/content/val2017.zip",
    "/content/annotations_trainval2017.zip"
]:
    if os.path.exists(f):
        os.remove(f)
        print("Deleted:", f)

Deleted: /content/val2017.zip
Deleted: /content/annotations_trainval2017.zip


In [19]:
import json
import os
from collections import defaultdict

# ==========================================
# JetAuto 15 classes
# ==========================================

JETAUTO_CLASSES = {
    0: "person",
    1: "bicycle",
    2: "car",
    3: "motorcycle",
    4: "bus",
    5: "truck",
    6: "traffic light",
    7: "fire hydrant",
    8: "stop sign",
    9: "bench",
    10: "chair",
    11: "couch",
    12: "potted plant",
    13: "clock",
    14: "tv"
}

# COCO ID -> JetAuto ID
COCO_TO_JETAUTO = {
    1: 0,    # person
    2: 1,    # bicycle
    3: 2,    # car
    4: 3,    # motorcycle
    6: 4,    # bus
    8: 5,    # truck
    10: 6,   # traffic light
    11: 7,   # fire hydrant
    13: 8,   # stop sign
    15: 9,   # bench
    62: 10,  # chair
    63: 11,  # couch
    64: 12,  # potted plant
    85: 13,  # clock
    72: 14   # tv
}

# ==========================================
# Paths
# ==========================================

COCO_JSON = "/content/annotations/instances_train2017.json"
COCO_IMAGES = "/content/datasets/coco/images/train2017"

LABEL_DIR = "/content/jetauto15/labels/train"

os.makedirs(LABEL_DIR, exist_ok=True)

# ==========================================
# Load COCO annotations
# ==========================================

print("Loading COCO training annotations...")

with open(COCO_JSON, "r") as f:
    coco = json.load(f)

print("COCO training images:", len(coco["images"]))
print("COCO annotations:", len(coco["annotations"]))

# ==========================================
# Image information
# ==========================================

images = {
    img["id"]: img
    for img in coco["images"]
}

# ==========================================
# Collect only our 15 classes
# ==========================================

image_annotations = defaultdict(list)

for ann in coco["annotations"]:

    if ann["category_id"] in COCO_TO_JETAUTO:
        image_annotations[ann["image_id"]].append(ann)

print(
    "Images containing JetAuto classes:",
    len(image_annotations)
)

# ==========================================
# Convert COCO bbox → YOLO bbox
# ==========================================

total_objects = 0
total_images = 0

for image_id, anns in image_annotations.items():

    img = images[image_id]

    width = img["width"]
    height = img["height"]

    label_name = os.path.splitext(
        img["file_name"]
    )[0] + ".txt"

    label_path = os.path.join(
        LABEL_DIR,
        label_name
    )

    with open(label_path, "w") as f:

        for ann in anns:

            x, y, w, h = ann["bbox"]

            # COCO → YOLO normalized coordinates
            x_center = (x + w / 2) / width
            y_center = (y + h / 2) / height

            w_norm = w / width
            h_norm = h / height

            class_id = COCO_TO_JETAUTO[
                ann["category_id"]
            ]

            f.write(
                f"{class_id} "
                f"{x_center:.6f} "
                f"{y_center:.6f} "
                f"{w_norm:.6f} "
                f"{h_norm:.6f}\n"
            )

            total_objects += 1

    total_images += 1

print("\n==============================")
print("TRAINING LABELS CREATED")
print("==============================")
print("Images with selected classes:", total_images)
print("Objects:", total_objects)
print("Labels directory:", LABEL_DIR)

Loading COCO training annotations...
COCO training images: 118287
COCO annotations: 860001
Images containing JetAuto classes: 82803

TRAINING LABELS CREATED
Images with selected classes: 82803
Objects: 429843
Labels directory: /content/jetauto15/labels/train


In [20]:
import os

label_dir = "/content/jetauto15/labels/train"

print("Training labels:",
      len(os.listdir(label_dir)))

print("Validation labels:",
      len(os.listdir("/content/jetauto15/labels/val")))

Training labels: 82803
Validation labels: 3527


In [21]:
import yaml

data = {
    "path": "/content",

    "train": "/content/datasets/coco/images/train2017",
    "val": "/content/jetauto15/images/val",

    "names": {
        0: "person",
        1: "bicycle",
        2: "car",
        3: "motorcycle",
        4: "bus",
        5: "truck",
        6: "traffic light",
        7: "fire hydrant",
        8: "stop sign",
        9: "bench",
        10: "chair",
        11: "couch",
        12: "potted plant",
        13: "clock",
        14: "tv"
    }
}

yaml_path = "/content/jetauto15.yaml"

with open(yaml_path, "w") as f:
    yaml.dump(data, f, sort_keys=False)

print(open(yaml_path).read())

path: /content
train: /content/datasets/coco/images/train2017
val: /content/jetauto15/images/val
names:
  0: person
  1: bicycle
  2: car
  3: motorcycle
  4: bus
  5: truck
  6: traffic light
  7: fire hydrant
  8: stop sign
  9: bench
  10: chair
  11: couch
  12: potted plant
  13: clock
  14: tv



In [22]:
import os

base = "/content/jetauto15"

os.makedirs(f"{base}/images", exist_ok=True)

# Remove existing train link if necessary
train_link = f"{base}/images/train"

if os.path.islink(train_link):
    os.unlink(train_link)
elif os.path.exists(train_link):
    print("WARNING: train directory already exists")

# Create symbolic link to original COCO images
os.symlink(
    "/content/datasets/coco/images/train2017",
    train_link
)

print("Training image link created:")
print(train_link)

print("\nPoints to:")
print(os.readlink(train_link))

Training image link created:
/content/jetauto15/images/train

Points to:
/content/datasets/coco/images/train2017


In [23]:
labels_link = "/content/jetauto15/labels/train"

print("Training labels:", os.path.exists(labels_link))
print("Number of labels:", len(os.listdir(labels_link)))

Training labels: True
Number of labels: 82803


In [27]:
import os

coco_label_root = "/content/datasets/coco/labels"
jetauto_train_labels = "/content/jetauto15/labels/train"

os.makedirs(coco_label_root, exist_ok=True)

target = "/content/datasets/coco/labels/train2017"

# Remove an old link if one exists
if os.path.islink(target):
    os.unlink(target)

# Create link to our JetAuto 15-class labels
os.symlink(jetauto_train_labels, target)

print("Created:")
print(target)

print("Points to:")
print(os.readlink(target))

print("Training labels:",
      len(os.listdir(jetauto_train_labels)))

Created:
/content/datasets/coco/labels/train2017
Points to:
/content/jetauto15/labels/train
Training labels: 82803


In [28]:
import yaml

data = {
    "path": "/content/jetauto15",

    "train": "images/train",
    "val": "images/val",

    "names": [
        "person",
        "bicycle",
        "car",
        "motorcycle",
        "bus",
        "truck",
        "traffic light",
        "fire hydrant",
        "stop sign",
        "bench",
        "chair",
        "couch",
        "potted plant",
        "clock",
        "tv"
    ]
}

yaml_path = "/content/jetauto15.yaml"

with open(yaml_path, "w") as f:
    yaml.dump(data, f, sort_keys=False)

print(open(yaml_path).read())

path: /content/jetauto15
train: images/train
val: images/val
names:
- person
- bicycle
- car
- motorcycle
- bus
- truck
- traffic light
- fire hydrant
- stop sign
- bench
- chair
- couch
- potted plant
- clock
- tv



In [29]:
from ultralytics.data.utils import check_det_dataset

check_det_dataset("/content/jetauto15.yaml")

{'path': PosixPath('/content/jetauto15'),
 'train': '/content/datasets/coco/images/train2017',
 'val': '/content/jetauto15/images/val',
 'names': {0: 'person',
  1: 'bicycle',
  2: 'car',
  3: 'motorcycle',
  4: 'bus',
  5: 'truck',
  6: 'traffic light',
  7: 'fire hydrant',
  8: 'stop sign',
  9: 'bench',
  10: 'chair',
  11: 'couch',
  12: 'potted plant',
  13: 'clock',
  14: 'tv'},
 'yaml_file': '/content/jetauto15.yaml',
 'nc': 15,
 'channels': 3}

In [39]:
!wget --no-check-certificate \
  -O /content/train2017.zip \
  http://images.cocodataset.org/zips/train2017.zip

--2026-09-04 05:25:49--  http://images.cocodataset.org/zips/train2017.zip
Resolving images.cocodataset.org (images.cocodataset.org)... 52.217.226.41, 16.15.253.195, 16.15.236.138, ...
Connecting to images.cocodataset.org (images.cocodataset.org)|52.217.226.41|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 19336861798 (18G) [application/zip]
Saving to: ‘/content/train2017.zip’

/content/train2017. 100%[===================>]  18.01G  46.9MB/s    in 8m 45s  

2026-09-04 05:34:34 (35.1 MB/s) - ‘/content/train2017.zip’ saved [19336861798/19336861798]



In [45]:
!mkdir -p /content/datasets/coco/images
!unzip -q /content/train2017.zip -d /content/datasets/coco/images

In [46]:


import os
import shutil

# Check whether the COCO training ZIP is still somewhere in /content
for root, dirs, files in os.walk("/content"):
    for f in files:
        if "train2017" in f.lower() and f.lower().endswith(".zip"):
            path = os.path.join(root, f)
            print("Found:", path)
            print("Size (GB):", round(os.path.getsize(path) / (1024**3), 2))

Found: /content/train2017.zip
Size (GB): 18.01


In [47]:
import os

print("Train images:", os.path.exists("/content/datasets/coco/images/train2017"))
print("Train labels:", os.path.exists("/content/datasets/coco/labels/train2017"))
print("Val images:", os.path.exists("/content/jetauto15/images/val"))
print("Val labels:", os.path.exists("/content/jetauto15/labels/val"))

Train images: True
Train labels: True
Val images: True
Val labels: True


In [48]:
!df -h /content

Filesystem      Size  Used Avail Use% Mounted on
overlay         113G   87G   27G  77% /


In [49]:
from ultralytics import YOLO

# Load pretrained YOLO11n
model = YOLO("yolo11n.pt")

# Small test run
results = model.train(
    data="/content/jetauto15.yaml",
    epochs=1,
    imgsz=640,
    batch=16,
    device=0,
    workers=2,
    project="/content/jetauto_results",
    name="yolo11n_test",
    exist_ok=True
)

Ultralytics 8.4.138 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/jetauto15.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=1, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolo11n_test, nbs=64, nms=False, opset=N